<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Zero-cost abstraction: classes and polymorphism compile away

A `@cute.kernel` body is **Python that runs at trace time** (the metaprogram idea from
*02_control_flow*). So the abstractions you reach for — classes, objects, methods, even
*polymorphism* — are resolved by the Python interpreter while the kernel is built, and compile
down to **nothing extra**. The device program is exactly the arithmetic and memory ops you emit.
We show two examples and read the generated PTX to prove the abstractions are free.

**You'll learn:** that ordinary Python **classes / objects / methods** used inside a kernel are
**zero-cost** (resolved at trace time, never present on the device); that a **polymorphic pipeline**
— a list of objects sharing a (virtual) method — *unrolls and inlines* into straight-line fused
arithmetic; and how to read the generated **PTX** (`cute.compile[KeepPTX(True)](...).__ptx__`) to
confirm there is no object, no vtable, and no dispatch.

**Runs on:** any CUDA GPU.

In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute import KeepPTX   # lets us read the generated PTX back via __ptx__
from typing import Any
import torch

## 1. A class is free

`Printer` is an ordinary Python class. Inside the kernel we build one and call its `.print(val)`
method **twice** — once naming the class directly, once through a class handed in as a `Constexpr`
argument. Both are pure Python, run while the host traces the body; the only thing *emitted* is the
`cute.printf(val)` inside `.print`.

In [ ]:
class Printer:
    def __init__(self, printer_method):
        self.printer_method = printer_method

    def print(self, v):
        self.printer_method(v)


@cute.kernel
def printer_kernel(class_type: cutlass.Constexpr[Any], val: cutlass.Int32):
    pm = cute.printf
    Printer(pm).print(val)        # construct a class, call its method
    class_type(pm).print(val)     # same, via a class passed in as Constexpr


@cute.jit
def printer(class_type: cutlass.Constexpr[Any], val: cutlass.Int32):
    printer_kernel(class_type, val).launch(grid=[1, 1, 1], block=[1, 1, 1])

Run it, then read the PTX. The whole device program is **two `vprintf` calls** (one per `.print`).
`Printer` survives only inside the *mangled kernel name* — it was a compile-time `Constexpr` baked
into the kernel's identity, never an object on the device.

In [ ]:
cutlass.cuda.initialize_cuda_context()
printer(Printer, 1)
cutlass.cuda.stream_sync(cutlass.cuda.default_stream())   # -> prints 1 twice

compiled = cute.compile[KeepPTX(True)](printer, Printer, 1)
ptx = compiled.__ptx__
print(ptx)
print("\n>>> vprintf call sites:", sum(1 for l in ptx.splitlines() if 'call' in l and 'vprintf' in l))

# Expected: the device program is just
#   .visible .entry kernel_cutlass_..._class___main__Printer__0( ... )  <- 'Printer' only in the baked name
#   call.uni (retval0), vprintf, (...);   <- .print(val) #1
#   call.uni (retval0), vprintf, (...);   <- .print(val) #2
# >>> vprintf call sites: 2
# 1
# 1

## 2. Polymorphism is free

Now the real thing: a chain of elementwise ops, written as a small class hierarchy with a *virtual*
`apply()` — a base `Op` and `AddN` / `MulN` overriding it. The kernel walks a **build-time list** of
op objects and calls `op.apply(x)` on each. Both the list and the `for` loop are metaprogramming:
the loop unrolls at trace time and each virtual `apply()` resolves in Python and inlines — leaving
straight-line fused arithmetic, with no object, no vtable, no dispatch.

In [ ]:
class Op:
    def apply(self, x):
        raise NotImplementedError      # 'virtual' -- overridden below


class AddN(Op):
    def __init__(self, n):
        self.n = n

    def apply(self, x):
        return x + self.n


class MulN(Op):
    def __init__(self, k):
        self.k = k

    def apply(self, x):
        return x * self.k


@cute.kernel
def fused_kernel(arr: cutlass.Array, ops: cutlass.Constexpr[Any]):
    tx, _, _ = cute.arch.thread_idx()
    x = arr[tx]
    for op in ops:          # build-time list -> loop UNROLLS at trace time
        x = op.apply(x)     # virtual call resolves in Python -> op INLINES
    arr[tx] = x             # -> straight-line fused arithmetic in the IR


@cute.jit
def run(arr: cutlass.Array, ops: cutlass.Constexpr[Any]):
    fused_kernel(arr, ops).launch(grid=[1, 1, 1], block=[256, 1, 1])

Build the pipeline with ordinary Python — a list of polymorphic ops — run it, and check the result.
Then read the fused kernel's PTX: the three `apply()` calls have become a couple of arithmetic
instructions with **zero `call` sites**. Swap or extend the op list and the kernel respecializes for
free.

In [ ]:
ops = [AddN(5.0), MulN(2.0), AddN(-1.0)]   # (x + 5) * 2 - 1, built from objects
n = 256
data = torch.arange(n, dtype=torch.float32).cuda()
run(cute.runtime.from_dlpack(data), ops)
cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
ref = (torch.arange(n, dtype=torch.float32).cuda() + 5.0) * 2.0 - 1.0
torch.testing.assert_close(data, ref, atol=1e-3, rtol=1e-3)
print("pipeline PASS")

# Read the fused kernel's PTX -- the op pipeline collapsed to straight-line arithmetic.
data2 = torch.arange(n, dtype=torch.float32).cuda()
ptx = cute.compile[KeepPTX(True)](run, cute.runtime.from_dlpack(data2), ops).__ptx__
for l in ptx.splitlines():
    if any(k in l for k in ('ld.global', 'add.f32', 'mul.f32', 'fma.', 'st.global')):
        print("   ", l.strip())
print(">>> call sites in the fused kernel:", ptx.count('call'))

# Expected output: three ops -> two arithmetic instructions, no calls:
#   ld.global.b32   %r2, [%rd3];
#   add.f32         %r3, %r2, 0f40A00000;                  <- + 5.0
#   fma.rn.f32      %r4, %r3, 0f40000000, 0fBF800000;      <- * 2.0 - 1.0  (fused)
#   st.global.b32   [%rd3], %r4;
# >>> call sites in the fused kernel: 0
# pipeline PASS

## Takeaway

Three polymorphic `apply()` calls compiled to **two arithmetic instructions and zero `call`s**. The
`Op` hierarchy, the virtual dispatch, the Python `for` over the op list — all of it ran in the Python
interpreter at trace time and left no trace on the device. That is *zero-cost abstraction*: you write
the kernel with the full Python object model, and pay for none of it at runtime. (Same mechanism as
the meta loop in *02_control_flow* — the kernel body is a metaprogram — applied to objects.)

## Try it yourself

1. **Respecialize for free.** Change `ops` to `[MulN(3.0), AddN(7.0)]` and re-run — no kernel edit,
   just a different build-time list, and the PTX changes to match.
2. **Add an op.** Define `class Square(Op)` whose `apply` returns `x * x`, drop it into the list, and
   re-read the PTX: one more `mul.f32`, still **zero `call`s**.
3. **Watch the loop unroll.** Put a trace-time `print(type(op).__name__)` in the `for` loop — it
   fires once per op, in Python, while tracing (exactly like the meta loop in *02_control_flow*).
4. **Other artifacts.** `compiled.__sass__` shows the final SASS the same way `__ptx__` shows PTX.